In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [ ]:
class EncodeTextSource:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    self.load_codebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [ ]:
def categoricallabelAll(w):

  if w=="['Initial state']":
    return 0
  if w=="['Final state']":
    return 1
  if w=="['State transformation']":
    return 2
  if w=="['Initial state', 'Final state']":
    return 3
  if w=="['Initial state', 'State transformation']":
    return 4
  if w=="['Final state', 'State transformation']":
    return 5
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 6
  return 7

category=np.array([
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [ ]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [ ]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [ ]:
train_full = train_full[train_full['Etiqueta 1']!='Correct'].copy()

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
np.unique(train_full['Etiqueta 2'])

array(["['Final state', 'State transformation']", "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [ ]:
encoder=EncodeTextSource()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Solución'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()

CPU times: user 2min 30s, sys: 2.81 s, total: 2min 33s
Wall time: 2min 44s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((3459,), (3459,), (3459,), (3459,), (3459,))

In [ ]:
print(startstate.shape)
print(startstate[1262].shape)


(3459,)
(1, 9, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((3459, 768), (3459, 768), (3459, 768), (3459, 768))

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xcode_train,Xcode_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xcode,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [ ]:
Xp_train.shape,Xp_test.shape,Xcode_train.shape, Xcode_test.shape, Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767,),
 (692,))

# Keras model

In [ ]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x

def neural_network_all(problem,code,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p,mlp_code ,mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input,code,start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_code_gries(code_input,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_code, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input, start_input, trass_input,final_input], outputs=output)
  return model


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_wproblem(start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_problem_code(problem,code_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_code])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem, code_input], outputs=output)
  return model

def neural_network_code(code_input,base=10,pow_initial=4,num_max_blocks=3):

  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## output layer
  output=Dense(50)(mlp_code)
  output=Dense(50)(mlp_code)
  output=Dense(50)(mlp_code)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input], outputs=output)
  return model

In [ ]:
import numpy as np
np.linspace(5,70,14)


array([ 5., 10., 15., 20., 25., 30., 35., 40., 45., 50., 55., 60., 65.,
       70.])

In [ ]:
models=[]
models_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

problem_input=Input((768,))
code_input=Input((768,))
start_input=Input((768,))
trass_input=Input((768,))
final_input=Input((768,))

print("problem all")
models.append(neural_network_all(problem_input,code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem gries")
models.append(neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code gries")
models.append(neural_network_code_gries(code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("gries")
models.append(neural_network_wproblem(start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code")
models.append(neural_network_problem_code(problem_input,code_input,base=8,pow_initial=1, num_max_blocks=3))
print("code")
models.append(neural_network_code(code_input,base=8,pow_initial=1, num_max_blocks=3))


problem all
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem code gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
Bo

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from time import time
#tensorflow random state
tf.random.set_seed(2023)

times=[]
max_accuracies=[]
max_val_accuracies=[]
max_val_losses=[]
max_losses=[]
learning_rates=[]
Model_Xtrain=[Xp_train,Xcode_train,Xs_train,Xt_train,Xf_train]

for i,my_model in enumerate(models):
  print(models_names[i])

  problem_input=Input((768,))
  start_input=Input((768,))
  trass_input=Input((768,))
  final_input=Input((768,))

  patience=60

  my_model.compile(loss='sparse_categorical_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

  es = EarlyStopping(monitor='val_loss', mode='min', patience=patience)

  mc = ModelCheckpoint('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_finall2_{0}.keras'.format(models_names[i]), monitor='val_loss', mode='min', save_best_only=True)
  reduce_lr = ReduceLROnPlateau(
      monitor='val_loss',
      factor=0.1,
      patience=patience//2,
      min_lr=1e-9,
      verbose=1
  )
  if(i==1):
    Model_Xtrain=[Xp_train,Xs_train,Xt_train,Xf_train]
  if(i==2):
    Model_Xtrain=[Xcode_train,Xs_train,Xt_train,Xf_train]
  if(i==3):
    Model_Xtrain=[Xs_train,Xt_train,Xf_train]
  if(i==4):
    Model_Xtrain=[Xp_train,Xcode_train]
  if(i==5):
    Model_Xtrain=[Xcode_train]

  time1=time()
  history=my_model.fit(Model_Xtrain,y_train,batch_size=100,epochs=1000,
            validation_split=0.2,callbacks=[es, mc,reduce_lr])
  time2=time()

  df_histories=pd.DataFrame(history.history)
  df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_best_finall2_{0}.csv'.format(models_names[i]))

  times.append(time2-time1)

  max_accuracies.append(max(history.history['accuracy']))
  max_val_accuracies.append(max(history.history['val_accuracy']))
  max_val_losses.append(max(history.history['val_loss']))
  max_losses.append(max(history.history['loss']))
  learning_rates.append(min(history.history['learning_rate']))

  print(f"{patience}")
  print("times=",time2-time1)
  print("max_val_accuracy",max(history.history['val_accuracy']))
  print("max_accuracy",max(history.history['accuracy']))


  print("max_val_loss",max(history.history['val_loss']))
  print("max_loss",max(history.history['loss']))
  print("learning_rate",min(history.history['learning_rate']))





problemall
Epoch 1/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 38s 702ms/step - accuracy: 0.2340 - loss: 2.1849 - val_accuracy: 0.1516 - val_loss: 1.9739 - learning_rate: 0.0010
Epoch 2/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5462 - loss: 1.5219 - val_accuracy: 0.4603 - val_loss: 1.9109 - learning_rate: 0.0010
Epoch 3/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6512 - loss: 1.3349 - val_accuracy: 0.5307 - val_loss: 1.8321 - learning_rate: 0.0010
Epoch 4/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7122 - loss: 1.1957 - val_accuracy: 0.5072 - val_loss: 1.7184 - learning_rate: 0.0010
Epoch 5/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.7601 - loss: 1.0897 - val_accuracy: 0.5758 - val_loss: 1.6464 - learning_rate: 0.0010
Epoch 6/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7634 - loss: 1.0360 - val_accuracy: 0.4765 - val_loss: 1.6070 - learning_rate: 0.0010
Epoch 7/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.81

In [ ]:
df_histories=pd.DataFrame({"times":times,"max_accuracies":max_accuracies,"max_val_accuracies":max_val_accuracies,"max_val_losses":max_val_losses,"max_losses":max_losses,"learning_rates":learning_rates})
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_finall2.csv')

In [ ]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates
0,152.149544,1.000000,0.944043,1.973901,2.006667,0.00001
1,113.132209,1.000000,0.942238,1.874851,2.010382,0.00001
2,146.371440,1.000000,0.944043,1.907495,2.024658,0.00001
3,112.367363,0.999548,0.938628,2.147737,2.029650,0.00001
4,116.266728,0.999096,0.940433,2.058622,2.179343,0.00001
5,77.379857,0.995933,0.935018,3.985698,4.116018,0.00001


In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [ ]:
from tensorflow import keras
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

Model_Xtest=[Xp_test,Xcode_test,Xs_test,Xt_test,Xf_test]
for i,my_model in enumerate(models):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_finall2_{0}.keras'.format(models_names[i]))
  if(i==1):
    Model_Xtest=[Xp_test,Xs_test,Xt_test,Xf_test]
  if(i==2):
    Model_Xtest=[Xcode_test,Xs_test,Xt_test,Xf_test]
  if(i==3):
    Model_Xtest=[Xs_test,Xt_test,Xf_test]
  if(i==4):
    Model_Xtest=[Xp_test,Xcode_test]
  if(i==5):
    Model_Xtest=[Xcode_test]


  evaluate=model.evaluate(Model_Xtest,y_test)

  t1=time()
  y_pred=model.predict(Model_Xtest)
  t2=time()
  mcc=calculate_mcc_multiclass(y_test, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.9338 - loss: 0.2103
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step
accuracy 0.926300585269928
loss 0.26626649498939514
mcc 0.9017708124169711
auc_pr 0.901550772579258
time predict 2.052982807159424


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9352 - loss: 0.2125
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step
accuracy 0.9291907548904419
loss 0.2580889165401459
mcc 0.905745433684328
auc_pr 0.8923134281241505
time predict 2.346470832824707


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9464 - loss: 0.1990
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step
accuracy 0.9364162087440491
loss 0.2485520839691162
mcc 0.9150901649806642
auc_pr 0.901008839458012
time predict 1.5875868797302246


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9128 - loss: 0.4053
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
accuracy 0.9075144529342651
loss 0.45502886176109314
mcc 0.8769563125005919
auc_pr 0.8925518895740898
time predict 1.4774999618530273


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9054 - loss: 0.4435
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
accuracy 0.897398829460144
loss 0.4976902902126312
mcc 0.8631077264433537
auc_pr 0.5600727762634269
time predict 0.8003878593444824


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

In [ ]:
df_histories['accuracies_predict']=accuracies_predict
df_histories['loss_predict']=loss_predict
df_histories['times_predict']=times_predict
df_histories['mccs']=mccs_predict
df_histories['aucpr']=aucpr_predict
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/early_best_finall_predict2.csv')

In [ ]:
df_histories['model']=models_names

In [ ]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
0,152.149544,1.000000,0.944043,1.973901,2.006667,0.00001,0.926301,0.266266,2.052983,0.901771,0.901551,problemall
1,113.132209,1.000000,0.942238,1.874851,2.010382,0.00001,0.929191,0.258089,2.346471,0.905745,0.892313,problem_gries
2,146.371440,1.000000,0.944043,1.907495,2.024658,0.00001,0.936416,0.248552,1.587587,0.915090,0.901009,code_gries
3,112.367363,0.999548,0.938628,2.147737,2.029650,0.00001,0.927746,0.279119,2.096335,0.903577,0.890342,gries
4,116.266728,0.999096,0.940433,2.058622,2.179343,0.00001,0.907514,0.455029,1.477500,0.876956,0.892552,problem_code
5,77.379857,0.995933,0.935018,3.985698,4.116018,0.00001,0.897399,0.497690,0.800388,0.863108,0.560073,code


In [ ]:
df_histories[(df_histories['aucpr']>0.89)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
0,152.149544,1.000000,0.944043,1.973901,2.006667,0.00001,0.926301,0.266266,2.052983,0.901771,0.901551,problemall
1,113.132209,1.000000,0.942238,1.874851,2.010382,0.00001,0.929191,0.258089,2.346471,0.905745,0.892313,problem_gries
2,146.371440,1.000000,0.944043,1.907495,2.024658,0.00001,0.936416,0.248552,1.587587,0.915090,0.901009,code_gries
3,112.367363,0.999548,0.938628,2.147737,2.029650,0.00001,0.927746,0.279119,2.096335,0.903577,0.890342,gries
4,116.266728,0.999096,0.940433,2.058622,2.179343,0.00001,0.907514,0.455029,1.477500,0.876956,0.892552,problem_code


In [ ]:
df_histories[(df_histories['aucpr']>0.89)&(df_histories['max_accuracies']-df_histories['max_val_accuracies']<0.06)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
0,152.149544,1.000000,0.944043,1.973901,2.006667,0.00001,0.926301,0.266266,2.052983,0.901771,0.901551,problemall
1,113.132209,1.000000,0.942238,1.874851,2.010382,0.00001,0.929191,0.258089,2.346471,0.905745,0.892313,problem_gries
2,146.371440,1.000000,0.944043,1.907495,2.024658,0.00001,0.936416,0.248552,1.587587,0.915090,0.901009,code_gries
4,116.266728,0.999096,0.940433,2.058622,2.179343,0.00001,0.907514,0.455029,1.477500,0.876956,0.892552,problem_code


Early stoping 40 the best, add code + gries